# 06 · 服务化与部署（LangServe）

学会写链之后，下一步是把链变成 **可被其他程序调用的服务**。本章用 **LangServe**（基于 FastAPI）把链暴露为 HTTP API。

运行前请确保：
- 已激活 `.venv` 虚拟环境
- 已确保 `requirements.txt` 中 `langserve` / `fastapi` / `uvicorn` 已安装（默认已启用）
- 已配置 `.env` 中的 `OPENAI_API_KEY`

> 详细概念讲解见 `docs/06-langserve-and-deployment.md`。


## 0. 环境检查


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv('OPENAI_API_KEY'):
    raise SystemExit('✗ 未找到 OPENAI_API_KEY，请复制 .env.example 为 .env 并填写。')
print('✓ 环境就绪，API Key 已加载')


## 1. 构造要发布的链与 FastAPI 应用

`add_routes(app, runnable, path="/chain")` 会把任意 `Runnable` 自动变成 REST API + 交互式 Playground。


In [ ]:
from fastapi import FastAPI
from langserve import add_routes
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))

# 要发布的链
chain = (
    ChatPromptTemplate.from_template('用一句话回答：{question}')
    | llm
    | StrOutputParser()
)

app = FastAPI(title='LangChain 学习 API', version='0.1.0')
add_routes(app, chain, path='/chain')


## 2. 直接验证链可用（无需启动服务器）

在部署前，先确认链本身能正常工作：


In [ ]:
print(chain.invoke({'question': 'LangChain 是什么？'}))


## 3. 启动服务

在终端（激活虚拟环境后）运行 **本 notebook 所在项目的** `examples/06_langserve.py`：

```bash
python examples/06_langserve.py
```

服务启动后，LangServe 自动生成以下接口：
- `GET  /chain/playground`：可视化调试界面
- `POST /chain/invoke`：单次调用
- `POST /chain/batch`：批量调用
- `POST /chain/stream`：流式输出
- `GET  /chain/input_schema`、`/chain/output_schema`：接口契约


## 4. 调用已启动的服务（可选）

服务启动后，可用 `requests` 或 `curl` 调用。下面演示用 `requests` 访问 `POST /chain/invoke`：


In [ ]:
# 注意：需先按第 3 步启动服务，再运行本单元格
import requests

try:
    resp = requests.post(
        'http://localhost:8000/chain/invoke',
        json={'input': {'question': '什么是 RAG？'}},
        timeout=30,
    )
    resp.raise_for_status()
    print('服务返回：', resp.json()['output'])
except Exception as e:
    print('调用失败（请确认服务已启动）：', e)


## 5. 完成

恭喜！你已经走完了 LangChain 的完整学习路线（环境 → 模型/提示词 → 链 → 记忆 → RAG → 代理 → 服务化）。

**常见坑**：`langserve`/`fastapi`/`uvicorn` 未安装；链的输入输出与注册 path 不匹配导致 422（用 `/input_schema` 核对）；生产部署需自行加鉴权、限流、健康检查。
